<a href="https://colab.research.google.com/github/Hschimera79/peptide-nasal-delivery-screen/blob/main/02_sequence_space.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Represent desmopressin as a list of residue dictionaries, keeping stereochemistry and modifications explicit
desmopressin = [
    {"pos": 1, "code": "Mpa", "chirality": None, "note": "deaminated Cys, disulfide partner of pos 6"},
    {"pos": 2, "code": "Tyr", "chirality": "L",  "note": "receptor binding"},
    {"pos": 3, "code": "Phe", "chirality": "L",  "note": "receptor binding"},
    {"pos": 4, "code": "Gln", "chirality": "L",  "note": ""},
    {"pos": 5, "code": "Asn", "chirality": "L",  "note": ""},
    {"pos": 6, "code": "Cys", "chirality": "L",  "note": "disulfide partner of pos 1"},
    {"pos": 7, "code": "Pro", "chirality": "L",  "note": ""},
    {"pos": 8, "code": "Arg", "chirality": "D",  "note": "D-form, key stability modification"},
    {"pos": 9, "code": "Gly", "chirality": "L",  "note": "C-terminal amide"},
]

for res in desmopressin:
    print(res["pos"], res["code"], res["chirality"])

1 Mpa None
2 Tyr L
3 Phe L
4 Gln L
5 Asn L
6 Cys L
7 Pro L
8 Arg D
9 Gly L


In [ ]:
# Define which positions may be mutated and which must stay fixed for structure or binding reasons
FIXED_POSITIONS = {
    1: "disulfide bridge",
    6: "disulfide bridge",
    2: "critical for V2 receptor binding",
    3: "critical for V2 receptor binding",
}

mutable = [r["pos"] for r in desmopressin if r["pos"] not in FIXED_POSITIONS]
print("Mutable positions:", mutable)

Mutable positions: [4, 5, 7, 8, 9]


In [ ]:
# Build the set of single-point mutants over the mutable positions only
AMINO_ACIDS = ["Ala", "Gly", "Ser", "Thr", "Val", "Leu", "Ile", "Pro",
               "Phe", "Tyr", "Trp", "Asn", "Gln", "Asp", "Glu",
               "Lys", "Arg", "His", "Met", "Cys"]

def make_variant(base, pos, new_code, new_chirality):
    variant = [dict(r) for r in base]
    for res in variant:
        if res["pos"] == pos:
            res["code"] = new_code
            res["chirality"] = new_chirality
            res["note"] = "mutated"
    return variant

variants = []
for pos in mutable:
    for aa in AMINO_ACIDS:
        original = next(r for r in desmopressin if r["pos"] == pos)
        if aa == original["code"]:
            continue
        variants.append({
            "id": f"{original['code']}{pos}{aa}",
            "parent": "desmopressin",
            "change": f"{original['code']}{pos} -> {aa}",
            "sequence": make_variant(desmopressin, pos, aa, "L"),
        })

print("Number of single-point variants:", len(variants))

Number of single-point variants: 95


In [ ]:
# Add stability-oriented variants: D-amino acid substitutions at mutable positions
d_variants = []
for pos in mutable:
    original = next(r for r in desmopressin if r["pos"] == pos)
    if original["code"] == "Gly":
        continue
    if original["chirality"] == "D":
        continue
    d_variants.append({
        "id": f"D-{original['code']}{pos}",
        "parent": "desmopressin",
        "change": f"L-{original['code']}{pos} -> D-{original['code']}{pos}",
        "sequence": make_variant(desmopressin, pos, original["code"], "D"),
    })

print("Number of D-substitution variants:", len(d_variants))

Number of D-substitution variants: 3


In [ ]:
# Merge all variants into a single table for downstream descriptor calculation
import pandas as pd

all_variants = variants + d_variants
df = pd.DataFrame([{"id": v["id"], "parent": v["parent"], "change": v["change"]} for v in all_variants])

print(df.shape)
df.head(10)

(98, 3)


,id,parent,change
0,Gln4Ala,desmopressin,Gln4 -> Ala
1,Gln4Gly,desmopressin,Gln4 -> Gly
2,Gln4Ser,desmopressin,Gln4 -> Ser
3,Gln4Thr,desmopressin,Gln4 -> Thr
4,Gln4Val,desmopressin,Gln4 -> Val
5,Gln4Leu,desmopressin,Gln4 -> Leu
6,Gln4Ile,desmopressin,Gln4 -> Ile
7,Gln4Pro,desmopressin,Gln4 -> Pro
8,Gln4Phe,desmopressin,Gln4 -> Phe
9,Gln4Tyr,desmopressin,Gln4 -> Tyr


In [ ]:
# Flag variants that are chemically risky for a disulfide-cyclized peptide
RISKY_RESIDUES = {
    "Pro": "may disrupt the cyclic backbone conformation",
    "Cys": "free thiol can scramble the existing disulfide bridge",
}

def risk_note(change):
    new_aa = change.split("-> ")[-1].strip()
    return RISKY_RESIDUES.get(new_aa, "")

df["risk"] = df["change"].apply(risk_note)
print(df["risk"].value_counts())
df[df["risk"] != ""].head()

risk
                                                         89
free thiol can scramble the existing disulfide bridge     5
may disrupt the cyclic backbone conformation              4
Name: count, dtype: int64


,id,parent,change,risk
7,Gln4Pro,desmopressin,Gln4 -> Pro,may disrupt the cyclic backbone conformation
18,Gln4Cys,desmopressin,Gln4 -> Cys,free thiol can scramble the existing disulfide...
26,Asn5Pro,desmopressin,Asn5 -> Pro,may disrupt the cyclic backbone conformation
37,Asn5Cys,desmopressin,Asn5 -> Cys,free thiol can scramble the existing disulfide...
56,Pro7Cys,desmopressin,Pro7 -> Cys,free thiol can scramble the existing disulfide...


In [ ]:
# Persist the variant table so the next module can start from it
df.to_csv("variants.csv", index=False)
print(df.shape)

(98, 4)
